# Benchmark & Analysis Engine - Bomberman 2D AI Pathfinding
**Proyek Penelitian Akademik:** Analisis Simulasi AI Musuh pada Game Bomberman 2D Berbasis Web Menggunakan Algoritma Pathfinding (BFS, Dijkstra, UCS, GBFS, A*)

Notebook ini dikembangkan khusus untuk melakukan eksperimen dan benchmarking kuantitatif secara otomatis guna memenuhi kebutuhan Proposal dan Penelitian:
1. **Perbandingan Efisiensi Pathfinding:** Mengevaluasi 5 algoritma (BFS, Dijkstra, UCS, GBFS, A*) berdasarkan *Nodes Explored*, *Execution Time (ms)*, dan *Path Length*.
2. **Variasi Kompleksitas Peta:** Membandingkan kinerja pada Peta Terbuka (*Open Map*) vs Peta Kompleks (*Complex Wood Walls Map*).
3. **Evaluasi Anti-Deadlock (History Penalty):** Mengukur rasio kemunculan *endless loop* pada interaksi Bot vs Bot sebelum dan sesudah penerapan *History Penalty*.
4. **Export Data:** Menghasilkan dataset CSV dan visualisasi grafik tingkat akademis untuk Bab 3 & Bab 4.

In [5]:
import time
import heapq
import random
from collections import deque
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Pengaturan visualisasi
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120

print('✅ Seluruh modul pendukung berhasil dimuat.')

ModuleNotFoundError: No module named 'matplotlib'

## 1. Definisi Peta Simulasi (Grid 15x13)
Keterangan elemen grid:
* `0` = Ubin Kosong / Jalan (Biaya = 1)
* `1` = Dinding Keras / Permanen (Tidak dapat dilewati)
* `2` = Dinding Kayu / Soft Wall (Biaya = 6, merepresentasikan dinding yang harus dihancurkan)

In [ ]:
MAP_OPEN = [
    [1,1,1,1,1,1,1,1,1,1,1,1,1,1,1],
    [1,0,0,0,0,0,0,0,0,0,0,0,0,0,1],
    [1,0,1,0,1,0,1,0,1,0,1,0,1,0,1],
    [1,0,0,0,0,0,0,0,0,0,0,0,0,0,1],
    [1,0,1,0,1,0,1,0,1,0,1,0,1,0,1],
    [1,0,0,0,0,0,0,0,0,0,0,0,0,0,1],
    [1,0,1,0,1,0,1,0,1,0,1,0,1,0,1],
    [1,0,0,0,0,0,0,0,0,0,0,0,0,0,1],
    [1,0,1,0,1,0,1,0,1,0,1,0,1,0,1],
    [1,0,0,0,0,0,0,0,0,0,0,0,0,0,1],
    [1,0,1,0,1,0,1,0,1,0,1,0,1,0,1],
    [1,0,0,0,0,0,0,0,0,0,0,0,0,0,1],
    [1,1,1,1,1,1,1,1,1,1,1,1,1,1,1]
]

MAP_COMPLEX = [
    [1,1,1,1,1,1,1,1,1,1,1,1,1,1,1],
    [1,0,0,2,2,2,2,2,2,2,2,2,0,0,1],
    [1,0,1,2,1,2,1,2,1,2,1,2,1,0,1],
    [1,2,2,2,2,2,2,2,2,2,2,2,2,2,1],
    [1,2,1,2,1,2,1,2,1,2,1,2,1,2,1],
    [1,2,2,2,2,0,2,2,2,2,2,2,2,2,1],
    [1,2,1,2,1,2,1,2,1,2,1,2,1,2,1],
    [1,2,2,2,2,2,2,2,2,2,2,2,2,2,1],
    [1,2,1,2,1,2,1,2,1,2,1,2,1,2,1],
    [1,2,2,2,2,2,2,2,2,2,2,2,2,2,1],
    [1,0,1,2,1,2,1,2,1,2,1,2,1,0,1],
    [1,0,0,2,2,2,2,2,2,2,2,2,0,0,1],
    [1,1,1,1,1,1,1,1,1,1,1,1,1,1,1]
]

print(f'Peta Terbuka dimuat: {len(MAP_OPEN)}x{len(MAP_OPEN[0])} sel')
print(f'Peta Kompleks dimuat: {len(MAP_COMPLEX)}x{len(MAP_COMPLEX[0])} sel')

## 2. Implementasi Algoritma Pathfinding & History Penalty

In [ ]:
def manhattan_distance(p1, p2):
    return abs(p1[0] - p2[0]) + abs(p1[1] - p2[1])

def get_neighbors(cell, grid, history_counts=None):
    rows, cols = len(grid), len(grid[0])
    r, c = cell
    neighbors = []
    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    for dr, dc in directions:
        nr, nc = r + dr, c + dc
        if 0 <= nr < rows and 0 <= nc < cols:
            val = grid[nr][nc]
            penalty = 0
            if history_counts and (nr, nc) in history_counts:
                penalty = history_counts[(nr, nc)] * 5
            
            if val == 0:
                neighbors.append(((nr, nc), 1 + penalty))
            elif val == 2:
                neighbors.append(((nr, nc), 6 + penalty))
    return neighbors

def reconstruct_path(came_from, start, target):
    path = []
    curr = target
    while curr in came_from:
        path.append(curr)
        curr = came_from[curr]
    path.append(start)
    path.reverse()
    return path

# 1. Breadth-First Search (BFS)
def run_bfs(grid, start, target, history=None):
    t_start = time.perf_counter()
    history_counts = {}
    if history:
        for h in history:
            history_counts[tuple(h)] = history_counts.get(tuple(h), 0) + 1

    queue = deque([start])
    visited = {start}
    came_from = {}
    nodes_explored = 0

    while queue:
        curr = queue.popleft()
        nodes_explored += 1
        if curr == target:
            exec_time = (time.perf_counter() - t_start) * 1000
            path = reconstruct_path(came_from, start, target)
            return path, nodes_explored, exec_time, len(path)

        nbrs = get_neighbors(curr, grid, history_counts)
        nbrs.sort(key=lambda x: x[1])
        for nxt, _ in nbrs:
            if nxt not in visited:
                visited.add(nxt)
                came_from[nxt] = curr
                queue.append(nxt)

    exec_time = (time.perf_counter() - t_start) * 1000
    return [], nodes_explored, exec_time, 0

# 2. Algoritma Dijkstra
def run_dijkstra(grid, start, target, history=None):
    t_start = time.perf_counter()
    history_counts = {}
    if history:
        for h in history:
            history_counts[tuple(h)] = history_counts.get(tuple(h), 0) + 1

    pq = [(0, start)]
    g_score = {start: 0}
    came_from = {}
    nodes_explored = 0

    while pq:
        curr_g, curr = heapq.heappop(pq)
        nodes_explored += 1
        if curr == target:
            exec_time = (time.perf_counter() - t_start) * 1000
            path = reconstruct_path(came_from, start, target)
            return path, nodes_explored, exec_time, len(path)

        if curr_g > g_score.get(curr, float('inf')):
            continue

        for nxt, cost in get_neighbors(curr, grid, history_counts):
            new_g = curr_g + cost
            if new_g < g_score.get(nxt, float('inf')):
                g_score[nxt] = new_g
                came_from[nxt] = curr
                heapq.heappush(pq, (new_g, nxt))

    exec_time = (time.perf_counter() - t_start) * 1000
    return [], nodes_explored, exec_time, 0

# 3. Uniform Cost Search (UCS)
def run_ucs(grid, start, target, history=None):
    return run_dijkstra(grid, start, target, history)

# 4. Greedy Best-First Search (GBFS)
def run_gbfs(grid, start, target, history=None):
    t_start = time.perf_counter()
    history_counts = {}
    if history:
        for h in history:
            history_counts[tuple(h)] = history_counts.get(tuple(h), 0) + 1

    pq = [(manhattan_distance(start, target), start)]
    visited = {start}
    came_from = {}
    nodes_explored = 0

    while pq:
        _, curr = heapq.heappop(pq)
        nodes_explored += 1
        if curr == target:
            exec_time = (time.perf_counter() - t_start) * 1000
            path = reconstruct_path(came_from, start, target)
            return path, nodes_explored, exec_time, len(path)

        for nxt, cost in get_neighbors(curr, grid, history_counts):
            if nxt not in visited:
                visited.add(nxt)
                came_from[nxt] = curr
                priority = manhattan_distance(nxt, target) + (cost - 1)
                heapq.heappush(pq, (priority, nxt))

    exec_time = (time.perf_counter() - t_start) * 1000
    return [], nodes_explored, exec_time, 0

# 5. A* Search
def run_a_star(grid, start, target, history=None):
    t_start = time.perf_counter()
    history_counts = {}
    if history:
        for h in history:
            history_counts[tuple(h)] = history_counts.get(tuple(h), 0) + 1

    pq = [(manhattan_distance(start, target), 0, start)]
    g_score = {start: 0}
    came_from = {}
    nodes_explored = 0

    while pq:
        f, curr_g, curr = heapq.heappop(pq)
        nodes_explored += 1
        if curr == target:
            exec_time = (time.perf_counter() - t_start) * 1000
            path = reconstruct_path(came_from, start, target)
            return path, nodes_explored, exec_time, len(path)

        if curr_g > g_score.get(curr, float('inf')):
            continue

        for nxt, cost in get_neighbors(curr, grid, history_counts):
            new_g = curr_g + cost
            if new_g < g_score.get(nxt, float('inf')):
                g_score[nxt] = new_g
                came_from[nxt] = curr
                new_f = new_g + manhattan_distance(nxt, target)
                heapq.heappush(pq, (new_f, new_g, nxt))

    exec_time = (time.perf_counter() - t_start) * 1000
    return [], nodes_explored, exec_time, 0

print('✅ Seluruh algoritma (BFS, Dijkstra, UCS, GBFS, A*) berhasil dikonfigurasi.')

## 3. Eksekusi Benchmark Suite 1: Efisiensi Komputasi Pathfinding

In [ ]:
algorithms = {
    'BFS': run_bfs,
    'Dijkstra': run_dijkstra,
    'UCS': run_ucs,
    'GBFS': run_gbfs,
    'A*': run_a_star
}

maps = {
    'Peta Terbuka (Open)': MAP_OPEN,
    'Peta Kompleks (Complex Wood Walls)': MAP_COMPLEX
}

start_pos = (1, 1)
target_pos = (11, 13)
NUM_TRIALS = 50

results = []

for map_name, grid_map in maps.items():
    for algo_name, algo_func in algorithms.items():
        nodes_list, time_list, path_len_list = [], [], []
        for _ in range(NUM_TRIALS):
            path, nodes, exec_time, path_len = algo_func(grid_map, start_pos, target_pos)
            nodes_list.append(nodes)
            time_list.append(exec_time)
            path_len_list.append(path_len)
        
        results.append({
            'Map Type': map_name,
            'Algorithm': algo_name,
            'Nodes Explored (Avg)': np.mean(nodes_list),
            'Nodes Explored (Std)': np.std(nodes_list),
            'Execution Time (ms)': np.mean(time_list),
            'Path Length': np.mean(path_len_list),
            'Success Rate (%)': 100.0 if np.mean(path_len_list) > 0 else 0.0
        })

df_benchmark = pd.DataFrame(results)
print('=== TABEL HASIL BENCHMARK EFISIENSI PATHFINDING ===')
display(df_benchmark)

## 4. Eksekusi Benchmark Suite 2: Pengujian Anti-Deadlock (History Penalty)

In [ ]:
def simulate_bvb_loop(grid, algo1, algo2, use_history_penalty=True, max_steps=100):
    start1, start2 = (1, 1), (11, 13)
    pos1, pos2 = start1, start2
    hist1, hist2 = [], []
    
    positions_history = set()
    oscillating_steps = 0
    
    algorithms_map = {
        'BFS': run_bfs, 'Dijkstra': run_dijkstra, 'UCS': run_ucs,
        'GBFS': run_gbfs, 'A*': run_a_star
    }
    
    f1 = algorithms_map[algo1]
    f2 = algorithms_map[algo2]
    
    for step in range(max_steps):
        state = (pos1, pos2)
        if state in positions_history:
            oscillating_steps += 1
        else:
            positions_history.add(state)
            
        if manhattan_distance(pos1, pos2) <= 1:
            return False, step
            
        p1, _, _, _ = f1(grid, pos1, pos2, hist1 if use_history_penalty else None)
        if len(p1) > 1:
            pos1 = p1[1]
            hist1.append(pos1)
            if len(hist1) > 6: hist1.pop(0)
            
        p2, _, _, _ = f2(grid, pos2, pos1, hist2 if use_history_penalty else None)
        if len(p2) > 1:
            pos2 = p2[1]
            hist2.append(pos2)
            if len(hist2) > 6: hist2.pop(0)
            
    is_deadlock = oscillating_steps > 30
    return is_deadlock, max_steps

deadlock_experiments = []
pairs = [('A*', 'A*'), ('A*', 'BFS'), ('GBFS', 'BFS'), ('Dijkstra', 'A*')]

for pair in pairs:
    deadlocks_without, _ = zip(*[simulate_bvb_loop(MAP_OPEN, pair[0], pair[1], use_history_penalty=False) for _ in range(20)])
    deadlocks_with, _ = zip(*[simulate_bvb_loop(MAP_OPEN, pair[0], pair[1], use_history_penalty=True) for _ in range(20)])
    
    deadlock_experiments.append({
        'Matchup': f'{pair[0]} vs {pair[1]}',
        'Deadlock Rate Without History Penalty (%)': np.mean(deadlocks_without) * 100,
        'Deadlock Rate With History Penalty (%)': np.mean(deadlocks_with) * 100
    })

df_deadlock = pd.DataFrame(deadlock_experiments)
print('=== TABEL HASIL EVALUASI ANTI-DEADLOCK ===')
display(df_deadlock)

## 5. Visualisasi Grafis Hasil Pengujian

In [ ]:
# Grafik 1: Perbandingan Jumlah Node Dieksplorasi
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, map_type in enumerate(['Peta Terbuka (Open)', 'Peta Kompleks (Complex Wood Walls)']):
    sub_df = df_benchmark[df_benchmark['Map Type'] == map_type]
    sns.barplot(data=sub_df, x='Algorithm', y='Nodes Explored (Avg)', palette='Blues_d', ax=axes[idx])
    axes[idx].set_title(f'Nodes Explored - {map_type}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Algoritma')
    axes[idx].set_ylabel('Rata-rata Node Dieksplorasi')
    for p in axes[idx].patches:
        axes[idx].annotate(f'{p.get_height():.1f}', (p.get_x() + p.get_width() / 2., p.get_height()),
                           ha='center', va='center', xytext=(0, 5), textcoords='offset points', fontsize=9)

plt.tight_layout()
plt.show()

# Grafik 2: Perbandingan Waktu Eksekusi (ms)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, map_type in enumerate(['Peta Terbuka (Open)', 'Peta Kompleks (Complex Wood Walls)']):
    sub_df = df_benchmark[df_benchmark['Map Type'] == map_type]
    sns.barplot(data=sub_df, x='Algorithm', y='Execution Time (ms)', palette='Reds_d', ax=axes[idx])
    axes[idx].set_title(f'Execution Time (ms) - {map_type}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Algoritma')
    axes[idx].set_ylabel('Waktu Eksekusi (ms)')
    for p in axes[idx].patches:
        axes[idx].annotate(f'{p.get_height():.3f} ms', (p.get_x() + p.get_width() / 2., p.get_height()),
                           ha='center', va='center', xytext=(0, 5), textcoords='offset points', fontsize=9)

plt.tight_layout()
plt.show()

# Grafik 3: Tingkat Kebuntuan Sebelum vs Sesudah History Penalty
plt.figure(figsize=(9, 5))
df_deadlock_melted = df_deadlock.melt(id_vars=['Matchup'], 
                                       value_vars=['Deadlock Rate Without History Penalty (%)', 'Deadlock Rate With History Penalty (%)'],
                                       var_name='Kondisi', value_name='Deadlock Rate (%)')

df_deadlock_melted['Kondisi'] = df_deadlock_melted['Kondisi'].replace({
    'Deadlock Rate Without History Penalty (%)': 'Tanpa History Penalty',
    'Deadlock Rate With History Penalty (%)': 'Dengan History Penalty'
})

sns.barplot(data=df_deadlock_melted, x='Matchup', y='Deadlock Rate (%)', hue='Kondisi', palette=['#e74c3c', '#2ecc71'])
plt.title('Pengaruh History Penalty terhadap Tingkat Kebuntuan (Deadlock Rate)', fontsize=12, fontweight='bold')
plt.xlabel('Pertandingan Head-to-Head (Bot vs Bot)')
plt.ylabel('Tingkat Kebuntuan / Endless Loop (%)')
plt.ylim(0, 110)
plt.legend(title='Skema Fitur')
plt.tight_layout()
plt.show()

## 6. Export Dataset & Ringkasan Laporan

In [ ]:
df_benchmark.to_csv('benchmark_efficiency_results.csv', index=False)
df_deadlock.to_csv('benchmark_deadlock_results.csv', index=False)

print('✅ Data benchmark telah berhasil diekspor ke file CSV:')
print('1. benchmark_efficiency_results.csv')
print('2. benchmark_deadlock_results.csv')
print('Siap dilampirkan dan dianalisis untuk bab pengujian proposal/skripsi.')